<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/APAC_AGENTIC_GGUF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/frank-morales2020/AST/blob/main/Qwen3dot8_27B_topo_cbp_gguf.ipynb

## setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/*.gguf

In [ ]:
!apt-get update && !apt-get install -y git
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --force-reinstall --no-cache-dir git+https://github.com/abetlen/llama-cpp-python.git --verbose

In [ ]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


GPU Offload Supported: True


In [ ]:
import os
import struct
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/GGUF_Models")

def read_gguf_header(filepath):
    try:
        with open(filepath, "rb") as f:
            magic = f.read(4)
            if magic != b"GGUF":
                return {"error": "Invalid magic bytes"}
            version = struct.unpack("<I", f.read(4))[0]
            tensor_count = struct.unpack("<Q", f.read(8))[0]
            kv_count = struct.unpack("<Q", f.read(8))[0]
            return {
                "version": version,
                "tensor_count": tensor_count,
                "kv_count": kv_count
            }
    except Exception as e:
        return {"error": str(e)}

print(f"{'Model Filename':<52} | {'Size (GB)':<10} | {'GGUF Version':<12} | {'Status'}")
print("-" * 95)

total_size = 0
for file_path in sorted(model_dir.glob("*.gguf")):
    file_size = file_path.stat().st_size
    size_gb = file_size / (1024**3)
    total_size += file_size

    meta = read_gguf_header(file_path)
    if "error" not in meta:
        status = "Verified"
        version_str = f"v{meta['version']}"
    else:
        status = meta["error"]
        version_str = "N/A"

    print(f"{file_path.name:<52} | {size_gb:>8.2f}GB | {version_str:<12} | {status}")

print("-" * 95)
print(f"Total Portfolio Storage: {total_size / (1024**3):.2f} GB across {len(list(model_dir.glob('*.gguf')))} models.")

Model Filename                                       | Size (GB)  | GGUF Version | Status
-----------------------------------------------------------------------------------------------
certified-scorpion-topo-qwen-bf16.gguf               |    14.19GB | v3           | Verified
deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf |    69.83GB | v3           | Verified
deepseek_moe_16b_topocbp_q4_k_m.gguf                 |    10.11GB | v3           | Verified
gpt-oss-20b-topo-governed-Q4_K_M.gguf                |    14.72GB | v3           | Verified
llama-3.1-70b-topo-governed-q4_k_m.gguf              |    39.60GB | v3           | Verified
mixtral_8x22b_q4_k_m.gguf                            |    79.71GB | v3           | Verified
muse-glimmer-30b-card-faithful-Q4_K_M.gguf           |    15.77GB | v3           | Verified
nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf          |     4.97GB | v3           | Verified
qwen3.8-27b-topocbp-Q8_0.gguf                        |    26.63GB | v3       

## DL/ML - APAC GGUF

In [ ]:
import os
from llama_cpp import Llama

class HybridAPACAgentSystem:
    def __init__(self, model_dir="/content/drive/MyDrive/GGUF_Models"):
        print("Initializing Hybrid APAC Agent Ecosystem (95GB VRAM Custom Offloading)...")

        # 1. Router: DeepSeek MoE (Smallest ~10.11 GB) -> Full GPU Offload
        print("Loading Router: DeepSeek MoE (n_gpu_layers=-1)...")
        self.router = Llama(
            model_path=os.path.join(model_dir, "deepseek_moe_16b_topocbp_q4_k_m.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 2. Synthesizer: Qwen 3.8-27B (Medium ~26.63 GB) -> Full GPU Offload
        print("Loading Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...")
        self.synthesizer = Llama(
            model_path=os.path.join(model_dir, "qwen3.8-27b-topocbp-Q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 3. Multilingual Specialist: Sarvam-30B (~31.86 GB) -> Full GPU Offload
        print("Loading Multilingual: Sarvam-30B (n_gpu_layers=25)...")
        self.multilingual = Llama(
            model_path=os.path.join(model_dir, "sarvam-30b-topo-governed-q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 4. Foundational: Certified Scorpion Qwen (~14.19 GB) -> Full GPU Offload
        print("Loading Foundational: Certified Scorpion Qwen (n_gpu_layers=-1)...")
        self.foundational = Llama(
            model_path=os.path.join(model_dir, "certified-scorpion-topo-qwen-bf16.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 5. Deep Reasoner: DeepSeek-R1 70B (Biggest ~69.83 GB) -> Partial Offload (n_gpu_layers=35)
        print("Loading Deep Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...")
        self.reasoner = Llama(
            model_path=os.path.join(model_dir, "deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"),
            n_ctx=8192, n_gpu_layers=25, verbose=False
        )

        print("All 5 APAC models active with custom offloading profiles under Topo-CBP governance.")

    def route_task(self, query: str) -> str:
        prompt = (
            f"Classify the following query into one of five categories "
            f"[REASONING, SYNTHESIS, MULTILINGUAL, FOUNDATIONAL, GENERAL]:\n\n{query}\nCategory:"
        )
        res = self.router(prompt, max_tokens=20, temperature=0.1)
        category = res["choices"][0]["text"].strip().upper()
        for cat in ["REASONING", "SYNTHESIS", "MULTILINGUAL", "FOUNDATIONAL", "GENERAL"]:
            if cat in category:
                return cat
        return "SYNTHESIS"

    def execute_pipeline(self, query: str):
        task_type = self.route_task(query)
        print(f"\n[Router: DeepSeek MoE] Task classified as: {task_type}")

        if task_type == "REASONING":
            print("[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep reasoning...")
            prompt = f"<|im_start|>system\nYou are a rigorous logical reasoning agent.<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n<think>\n"
            res = self.reasoner(prompt, max_tokens=1536, temperature=0.5, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        elif task_type == "SYNTHESIS":
            print("[Agent: Qwen 3.8-27B (n_gpu_layers=25] Executing synthesis...")
            prompt = f"<|im_start|>system\nYou are an expert synthesis agent.<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.6, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        elif task_type == "MULTILINGUAL":
            print("[Agent: Sarvam-30B (n_gpu_layers=25)] Processing regional context...")
            prompt = f"System: You are an expert multilingual processor.\nUser: {query}\nAssistant:"
            res = self.multilingual(prompt, max_tokens=1024, temperature=0.7)
            return res["choices"][0]["text"].strip()

        elif task_type == "FOUNDATIONAL":
            print("[Agent: Certified Scorpion Qwen (n_gpu_layers=-1)] Applying instruction tuning...")
            prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.foundational(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        else:
            print("[Agent: Qwen Synthesizer] Handling general objective...")
            prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

# --- Execution Test ---
if __name__ == "__main__":
    system = HybridAPACAgentSystem()
    sample_query = "Prove the convergence of continual backpropagation under topological invariant locking."
    result = system.execute_pipeline(sample_query)

    print("\n==============================")
    print("FINAL HYBRID PIPELINE OUTPUT")
    print("==============================\n")
    print(result)

Initializing Hybrid APAC Agent Ecosystem (95GB VRAM Custom Offloading)...
Loading Router: DeepSeek MoE (n_gpu_layers=-1)...
Loading Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...
Loading Multilingual: Sarvam-30B (n_gpu_layers=25)...
Loading Foundational: Certified Scorpion Qwen (n_gpu_layers=-1)...
Loading Deep Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...
All 5 APAC models active with custom offloading profiles under Topo-CBP governance.

[Router: DeepSeek MoE] Task classified as: REASONING
[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep reasoning...

FINAL HYBRID PIPELINE OUTPUT

Okay, so I need to prove the convergence of continual backpropagation under topological invariant locking. Hmm, I'm not exactly sure what that means, but I'll try to break it down. 

First, I know that backpropagation is a common algorithm used in training neural networks. It's about adjusting the weights to minimize the error. Continual backpropagation might refer to updating the weights continuo

In [ ]:
!nvidia-smi

Sun Nov 15 00:21:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             88W /  600W |   92929MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Medical  - APAC GGUF

In [ ]:
import os
from llama_cpp import Llama

class PureMedicalAPACAgentSystem:
    def __init__(self, model_dir="/content/drive/MyDrive/GGUF_Models"):
        print("Initializing Pure Medical APAC Agent Ecosystem (95GB VRAM Custom Offloading)...")

        # 1. Router: DeepSeek MoE (Smallest ~10.11 GB) -> Full GPU Offload
        print("Loading Clinical Router: DeepSeek MoE (n_gpu_layers=-1)...")
        self.router = Llama(
            model_path=os.path.join(model_dir, "deepseek_moe_16b_topocbp_q4_k_m.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 2. Synthesizer: Qwen 3.8-27B (Medium ~26.63 GB) -> Partial Offload (n_gpu_layers=25)
        print("Loading Clinical Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...")
        self.synthesizer = Llama(
            model_path=os.path.join(model_dir, "qwen3.8-27b-topocbp-Q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 3. Multilingual Specialist: Sarvam-30B (~31.86 GB) -> Partial Offload (n_gpu_layers=25)
        print("Loading Regional Biomedical Specialist: Sarvam-30B (n_gpu_layers=25)...")
        self.multilingual = Llama(
            model_path=os.path.join(model_dir, "sarvam-30b-topo-governed-q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 4. Foundational: Certified Scorpion Qwen (~14.19 GB) -> Full GPU Offload
        print("Loading Foundational Clinical Model: Certified Scorpion Qwen (n_gpu_layers=-1)...")
        self.foundational = Llama(
            model_path=os.path.join(model_dir, "certified-scorpion-topo-qwen-bf16.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 5. Deep Reasoner: DeepSeek-R1 70B (Biggest ~69.83 GB) -> Partial Offload (n_gpu_layers=25)
        print("Loading Deep Clinical Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...")
        self.reasoner = Llama(
            model_path=os.path.join(model_dir, "deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"),
            n_ctx=8192, n_gpu_layers=25, verbose=False
        )

        print("All 5 Medical APAC agent nodes active with custom offloading profiles under Topo-CBP governance.")

    def route_clinical_task(self, case_notes: str) -> str:
        prompt = (
            f"Classify the following clinical query into one of five categories "
            f"[DIFFERENTIAL_DIAGNOSIS, LITERATURE_ANALYSIS, PROTOCOL_SYNTHESIS, FOUNDATIONAL_REVIEW, GENERAL_CLINICAL]:\n\n{case_notes}\nCategory:"
        )
        res = self.router(prompt, max_tokens=25, temperature=0.1)
        category = res["choices"][0]["text"].strip().upper()
        for cat in ["DIFFERENTIAL_DIAGNOSIS", "LITERATURE_ANALYSIS", "PROTOCOL_SYNTHESIS", "FOUNDATIONAL_REVIEW", "GENERAL_CLINICAL"]:
            if cat in category:
                return cat
        return "DIFFERENTIAL_DIAGNOSIS"

    def execute_medical_pipeline(self, case_notes: str):
        task_type = self.route_clinical_task(case_notes)
        print(f"\n[Clinical Router: DeepSeek MoE] Task classified as: {task_type}")

        if task_type == "DIFFERENTIAL_DIAGNOSIS":
            print("[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep clinical differential reasoning...")
            prompt = (
                f"<|im_start|>system\nYou are an expert clinical reasoning assistant governed by strict biomedical validation.<|im_end|>\n"
                f"<|im_start|>user\nAnalyze the following patient presentation and provide a rigorous differential assessment:\n\n{case_notes}<|im_end|>\n"
                f"<|im_start|>assistant\n<think>\n"
            )
            res = self.reasoner(prompt, max_tokens=1536, temperature=0.5, stop=["<|im_end|>"])
            reasoning_output = res["choices"][0]["text"].strip()

            print("[Agent: Qwen 3.8-27B (n_gpu_layers=25)] Formatting structured clinical summary report...")
            synth_prompt = (
                f"<|im_start|>system\nYou are an expert medical writer and clinical synthesizer.<|im_end|>\n"
                f"<|im_start|>user\nBased on the following clinical reasoning, generate a clean, structured medical summary report:\n\n{reasoning_output}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
            #final_res = self.synthesizer(synth_prompt, max_tokens=1024, temperature=0.6, stop=["<|im_end|>"])
            final_res = self.synthesizer(synth_prompt, max_tokens=2048, temperature=0.6, stop=["<|im_end|>"])
            return final_res["choices"][0]["text"].strip()

        elif task_type == "LITERATURE_ANALYSIS":
            print("[Agent: Qwen 3.8-27B (n_gpu_layers=25)] Synthesizing biomedical literature and biomarkers...")
            prompt = (
                f"<|im_start|>system\nYou are a biomedical literature analysis assistant.<|im_end|>\n"
                f"<|im_start|>user\n{case_notes}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.6, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        elif task_type == "PROTOCOL_SYNTHESIS":
            print("[Agent: Sarvam-30B (n_gpu_layers=25)] Processing regional healthcare protocol and guidelines...")
            prompt = f"System: You are an expert clinical protocol processor.\nUser: {case_notes}\nAssistant:"
            res = self.multilingual(prompt, max_tokens=1024, temperature=0.7)
            return res["choices"][0]["text"].strip()

        elif task_type == "FOUNDATIONAL_REVIEW":
            print("[Agent: Certified Scorpion Qwen (n_gpu_layers=-1)] Applying foundational biomedical instruction review...")
            prompt = f"<|im_start|>user\n{case_notes}<|im_end|>\n<|im_start|>assistant\n"
            res = self.foundational(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        else:
            print("[Agent: Qwen Synthesizer] Handling general clinical query...")
            prompt = f"<|im_start|>user\n{case_notes}<|im_end|>\n<|im_start|>assistant\n"
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

# --- Execution Test ---
if __name__ == "__main__":
    medical_system = PureMedicalAPACAgentSystem()
    sample_clinical_query = "Patient presents with intermittent tachycardia, elevated serum ferritin, and microcytic anemia unresponsive to standard oral iron."
    report = medical_system.execute_medical_pipeline(sample_clinical_query)

    print("\n==============================")
    print("FINAL PURE MEDICAL PIPELINE REPORT")
    print("==============================\n")
    print(report)

Initializing Pure Medical APAC Agent Ecosystem (95GB VRAM Custom Offloading)...
Loading Clinical Router: DeepSeek MoE (n_gpu_layers=-1)...
Loading Clinical Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...
Loading Regional Biomedical Specialist: Sarvam-30B (n_gpu_layers=25)...
Loading Foundational Clinical Model: Certified Scorpion Qwen (n_gpu_layers=-1)...
Loading Deep Clinical Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...
All 5 Medical APAC agent nodes active with custom offloading profiles under Topo-CBP governance.

[Clinical Router: DeepSeek MoE] Task classified as: DIFFERENTIAL_DIAGNOSIS
[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep clinical differential reasoning...
[Agent: Qwen 3.8-27B (n_gpu_layers=25)] Formatting structured clinical summary report...

FINAL PURE MEDICAL PIPELINE REPORT

<think>
The user wants me to take the clinical reasoning provided and generate a clean, structured medical summary report. The reasoning is a detailed differential diagnosis workup f

In [ ]:
!nvidia-smi

Sun Nov 15 02:09:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   36C    P0             89W /  600W |   93087MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## EU/USA

In [ ]:
!nvidia-smi

Sun Nov 15 04:36:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             49W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip show markdown

Name: Markdown
Version: 3.11.2
Summary: Python implementation of John Gruber's Markdown.
Home-page: https://Python-Markdown.github.io/
Author: Manfred Stienstra, Yuri Takhteyev
Author-email: Waylan limberg <python.markdown@gmail.com>
License-Expression: BSD-3-Clause
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: panel, tensorboard


In [ ]:
"""
Financial multi-model pipeline (FINAL).

Models: GPT-OSS-20B (router), Llama 3.1 70B (synthesizer), Mixtral 8x22B (macro),
        Nemotron 3.5 Bolt (compliance), Muse Glimmer 30B (quant reasoner).

GPU distribution is the ORIGINAL one: all 5 models loaded concurrently with
  router -1, synthesizer 25, macro 25, foundation -1, reasoner -1 layers.

Safety nets:
  * The reasoner's raw output is saved to disk BEFORE synthesis, so a later failure loses nothing.
  * If synthesis fails or returns junk, the reasoner's answer is still saved as the report.
  * Timing is printed for every stage.

Setup:  pip install llama-cpp-python markdown
"""
import datetime
import gc
import html
import os
import re
import sys
import time

from llama_cpp import Llama, LlamaGrammar

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)

# ------------------------------------------------------------------ CONFIG
MODEL_DIR = "/content/drive/MyDrive/GGUF_Models"   # tip: copy to local disk for faster loads
OUT_DIR = "/content/drive/MyDrive/reports"
QUERY = "Model the impact of cross-border liquidity shocks on multi-currency portfolio Value-at-Risk."
PRELOAD_ALL = True    # your original setup: all 5 models resident at once
VERSION = "FINAL-v2"

CATEGORIES = [
    "QUANTITATIVE_MODELING",
    "PORTFOLIO_SYNTHESIS",
    "MACRO_MARKET_ANALYSIS",
    "REGULATORY_COMPLIANCE",
    "GENERAL_FINANCE",
]

# name -> (file, n_ctx, n_gpu_layers, extra kwargs)
MODELS = {
    "router":     ("gpt-oss-20b-topo-governed-Q4_K_M.gguf",       4096, -1, {}),
    "synth":      ("llama-3.1-70b-topo-governed-q4_k_m.gguf",     8192, 25, {}),
    "macro":      ("mixtral_8x22b_q4_k_m.gguf",                   4096, 25, {}),
    "foundation": ("nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf", 4096, -1, {}),
    "reasoner":   ("muse-glimmer-30b-card-faithful-Q4_K_M.gguf",  8192, -1, {}),
}

KEYWORDS = {
    "QUANTITATIVE_MODELING": [
        r"\bvar\b", r"value[- ]at[- ]risk", r"expected shortfall", r"\bmodel", r"\bderiv(e|ation)",
        r"simulat", r"monte carlo", r"volatil", r"covarian", r"correlation", r"stochastic",
        r"black[- ]scholes", r"\bgreeks\b", r"\bpricing\b",
    ],
    "PORTFOLIO_SYNTHESIS": [
        r"portfolio (optimi[sz]|construct|allocation)", r"asset allocation", r"rebalanc",
        r"efficient frontier",
    ],
    "MACRO_MARKET_ANALYSIS": [
        r"\bmacro", r"inflation", r"central bank", r"monetary policy", r"yield curve",
        r"\bgdp\b", r"market outlook",
    ],
    "REGULATORY_COMPLIANCE": [
        r"\bregulat", r"\bcompliance\b", r"\bbasel\b", r"\bmifid", r"dodd[- ]frank",
        r"\bsec\b", r"\baml\b", r"\bkyc\b", r"\baudit", r"sanction",
    ],
}

FORMAT_RULE = (
    "Write in plain Markdown with clear headings. Prefer plain-text formulas "
    "(e.g. VaR = z * sigma * sqrt(t) * V). If you must use LaTeX, wrap it in $...$ or $$...$$ only. "
    "Finish every section completely and never use placeholders such as '...' or '(rest of ...)'."
)


# ------------------------------------------------------------------ HELPERS
def log_time(label, t0):
    print(f"    [{label}: {time.time() - t0:.0f}s]")


def looks_valid(text):
    """Cheap sanity check so junk is never silently saved as a 'report'."""
    if len(text) < 1200:
        return False, "too short"
    if re.search(r"\(rest of|\[rest of|^\s*\.\.\.\s*$", text, re.I | re.M):
        return False, "contains a placeholder"
    wanted = ["objective", "assumptions", "methodology", "key findings", "risks", "next steps"]
    found = sum(1 for w in wanted if w in text.lower())
    if found < 4:
        return False, f"only {found}/6 expected sections found"
    return True, "ok"


MATH_RE = re.compile(
    r"(\$\$.+?\$\$|\\\[.+?\\\]|\\\(.+?\\\)|\$(?!\s)[^$\n]+?(?<!\s)\$)", re.S
)


def protect_math(md):
    store = []

    def sub(m):
        store.append(m.group(0))
        return f"@@MATH{len(store) - 1}@@"

    return MATH_RE.sub(sub, md), store


def restore_math(page_html, store):
    for i, m in enumerate(store):
        page_html = page_html.replace(f"@@MATH{i}@@", html.escape(m, quote=False))
    return page_html


MATHJAX = (
    "<script>window.MathJax={tex:{inlineMath:[['$','$'],['\\\\(','\\\\)']],"
    "displayMath:[['$$','$$'],['\\\\[','\\\\]']]}};</script>"
    "<script async src='https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js'></script>"
)


def save_report(query, task, body, note="", out_dir=OUT_DIR):
    """Save Markdown + HTML (formulas rendered with MathJax). Returns (md_path, html_path)."""
    os.makedirs(out_dir, exist_ok=True)
    now = datetime.datetime.now()
    base = os.path.join(out_dir, f"financial_report_{now:%Y%m%d_%H%M%S}")
    disclaimer = (
        "_AI-generated analysis for research purposes only. Not investment advice. "
        "Verify all figures, formulas and model outputs independently before relying on them._"
    )
    md = (
        f"# Financial Analysis Report\n\n"
        f"**Generated:** {now:%Y-%m-%d %H:%M}  \n"
        f"**Task type:** {task}  \n"
        f"**Query:** {query}\n"
        + (f"\n> **Note:** {note}\n" if note else "")
        + f"\n---\n\n{body}\n\n---\n\n{disclaimer}\n"
    )
    with open(base + ".md", "w", encoding="utf-8") as f:
        f.write(md)

    protected, store = protect_math(md)
    try:
        import markdown
        content = markdown.markdown(protected, extensions=["tables", "fenced_code"])
    except ImportError:
        content = "<pre style='white-space:pre-wrap'>" + html.escape(protected) + "</pre>"
    content = restore_math(content, store)

    page = (
        "<!doctype html><html><head><meta charset='utf-8'><title>Financial Report</title>"
        "<style>body{font-family:Georgia,serif;max-width:820px;margin:40px auto;padding:0 20px;"
        "line-height:1.6;color:#222}h1,h2,h3{font-family:Helvetica,Arial,sans-serif}"
        "table{border-collapse:collapse}td,th{border:1px solid #ccc;padding:6px 10px}"
        "blockquote{border-left:4px solid #e0a800;margin:1em 0;padding:.2em 1em;background:#fff8e1}"
        "</style>" + MATHJAX + "</head><body>" + content + "</body></html>"
    )
    with open(base + ".html", "w", encoding="utf-8") as f:
        f.write(page)
    return base + ".md", base + ".html"


# ------------------------------------------------------------------ SYSTEM
class FinancialAgentSystem:
    def __init__(self, model_dir=MODEL_DIR, preload_all=PRELOAD_ALL):
        self.model_dir = model_dir
        self.preload_all = preload_all
        self.loaded = {}
        print(f"financial_pipeline_final.py version {VERSION} | preload_all={preload_all}")
        names = list(MODELS) if preload_all else ["router"]
        for n in names:
            self._load(n)
        print("Ready.")

    def _load(self, name):
        if name not in self.loaded:
            fname, n_ctx, n_gpu, extra = MODELS[name]
            print(f"Loading {name}: {fname} (n_gpu_layers={n_gpu}, n_ctx={n_ctx})...")
            t0 = time.time()
            self.loaded[name] = Llama(
                model_path=os.path.join(self.model_dir, fname),
                n_ctx=n_ctx, n_gpu_layers=n_gpu, verbose=False, **extra,
            )
            log_time(f"loaded {name}", t0)
        return self.loaded[name]

    def _unload(self, name):
        if self.preload_all or name == "router":
            return
        if name in self.loaded:
            del self.loaded[name]
            gc.collect()

    @staticmethod
    def _chat(llm, system, user, max_tokens, temperature, **kw):
        res = llm.create_chat_completion(
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
            max_tokens=max_tokens, temperature=temperature, **kw,
        )
        ch = res["choices"][0]
        return ch["message"]["content"].strip(), ch.get("finish_reason")

    @staticmethod
    def _strip_think(text):
        if "</think>" in text:
            return text.split("</think>", 1)[1].strip(), True
        return re.sub(r"^\s*<think>\s*", "", text).strip(), False

    # ---------- routing
    def route(self, query):
        q = query.lower()
        hits = [c for c, pats in KEYWORDS.items() if any(re.search(p, q) for p in pats)]
        if len(hits) == 1:
            print(f"[Router] keyword rule matched: {hits[0]}")
            return hits[0]
        if hits:
            print(f"[Router] keywords matched {hits}; asking the LLM router")

        text = ""
        try:
            grammar = LlamaGrammar.from_string("root ::= " + " | ".join(f'"{c}"' for c in CATEGORIES))
            text, _ = self._chat(
                self._load("router"),
                "Classify the financial query into exactly one category. Definitions:\n"
                "- QUANTITATIVE_MODELING: risk models, VaR, derivations, simulations, pricing, statistics.\n"
                "- PORTFOLIO_SYNTHESIS: asset allocation, portfolio construction, rebalancing, optimization.\n"
                "- MACRO_MARKET_ANALYSIS: economies, central banks, rates, FX trends, market outlook.\n"
                "- REGULATORY_COMPLIANCE: laws, regulations, Basel/MiFID/SEC rules, reporting, audits.\n"
                "- GENERAL_FINANCE: everything else.\nReply with the category name only.",
                query, max_tokens=16, temperature=0.0, grammar=grammar,
            )
        except Exception as e:
            print(f"[Router] LLM routing failed: {e}")
        print(f"[Router] raw LLM answer: {text!r}")
        for c in CATEGORIES:
            if c in text.upper():
                return c
        print("[Router] no valid label; falling back to GENERAL_FINANCE")
        return "GENERAL_FINANCE"

    # ---------- pipeline
    def run(self, query, task=None):
        """Returns (task, report_body, note)."""
        task = task or self.route(query)
        print(f"\nTask classified as: {task}")
        t_all = time.time()

        if task == "QUANTITATIVE_MODELING":
            body, note = self._quant(query)
        elif task == "PORTFOLIO_SYNTHESIS":
            body, note = self._simple("synth", "You are an expert portfolio manager.", query, 2000), ""
        elif task == "MACRO_MARKET_ANALYSIS":
            body, note = self._simple("macro", "You are an expert global macroeconomic analyst.", query, 2000), ""
        elif task == "REGULATORY_COMPLIANCE":
            body, note = self._simple("foundation", "You are a financial regulatory compliance analyst.", query, 2000), ""
        else:
            body, note = self._simple("router", "You are a helpful finance assistant.", query, 1500), ""
        log_time("pipeline total", t_all)
        return task, body, note

    def _simple(self, model, role, query, max_tokens):
        print(f"[Agent: {model}] generating...")
        t0 = time.time()
        out, _ = self._chat(self._load(model), role + " " + FORMAT_RULE, query,
                            max_tokens=max_tokens, temperature=0.5)
        log_time("generate", t0)
        self._unload(model)
        return out

    def _quant(self, query):
        # ---- stage 1: deep reasoning (Muse Glimmer 30B)
        print("[Agent: Muse Glimmer 30B] quantitative reasoning...")
        t0 = time.time()
        raw, finish = self._chat(
            self._load("reasoner"),
            "You are an expert quantitative finance reasoning assistant. Be rigorous: state assumptions, "
            "and make sure every matrix or statistical construction is mathematically valid "
            "(e.g. covariance matrices stay symmetric and positive semi-definite). " + FORMAT_RULE,
            "Analyze the following and derive a rigorous quantitative framework "
            f"(assumptions, model, steps, risks):\n\n{query}",
            max_tokens=7000, temperature=0.5,   # reasoner n_ctx is 8192, so 7000 leaves room for the prompt
        )
        log_time("reasoning", t0)
        print(f"    [reasoner finish_reason: {finish}, output length: {len(raw)} chars]")
        answer, has_close = self._strip_think(raw)
        truncated = (finish == "length")
        note = ""
        if truncated:
            print("[Warning] Reasoner hit the token limit before finishing.")
            note = "The reasoning model ran out of tokens, so this report is based on partial reasoning."
            if not has_close:
                answer = raw[-6000:]   # unclosed <think> block: use the tail of the reasoning
        elif not answer:
            answer = raw.strip()       # finished normally but nothing after </think>: use everything
        self._unload("reasoner")

        # checkpoint so a later failure never loses the reasoning
        os.makedirs(OUT_DIR, exist_ok=True)
        ckpt = os.path.join(OUT_DIR, f"reasoning_{datetime.datetime.now():%Y%m%d_%H%M%S}.txt")
        with open(ckpt, "w", encoding="utf-8") as f:
            f.write(raw)
        print(f"    [Checkpoint saved: {ckpt}]")

        # ---- stage 2: structured report (Llama 3.1 70B)
        print("[Agent: Llama 3.1 70B] writing structured report...")
        # Synthesizer context is your original 4096 tokens: keep input (~1500 tokens) + output within it.
        answer_in = answer[:12000]
        prompts = [
            ("Turn the following quantitative reasoning into a professional structured report using "
             "these Markdown headings: ## Objective, ## Assumptions, ## Methodology, ## Key Findings, "
             "## Risks and Limitations, ## Next Steps.\n\n" + answer_in, 0.4),
            ("Write a COMPLETE professional report using these exact Markdown headings: "
             "## Objective, ## Assumptions, ## Methodology, ## Key Findings, ## Risks and Limitations, "
             "## Next Steps. Write full paragraphs under each heading. Source material:\n\n" + answer_in, 0.2),
        ]
        for attempt, (user_msg, temp) in enumerate(prompts, 1):
            try:
                t0 = time.time()
                out, _ = self._chat(
                    self._load("synth"),
                    "You are an expert financial analyst and report writer. " + FORMAT_RULE,
                    user_msg, max_tokens=2500, temperature=temp,
                )
                log_time(f"synthesis attempt {attempt}", t0)
            except Exception as e:
                print(f"[Warning] synthesis attempt {attempt} failed: {e}")
                continue
            ok, why = looks_valid(out)
            if ok:
                self._unload("synth")
                return out, note
            print(f"[Warning] synthesis attempt {attempt} rejected ({why}).")

        self._unload("synth")
        print("[Warning] Synthesis failed; using the reasoner's answer as the report.")
        return answer, (note + " Synthesis step failed; showing the reasoning model's answer directly.").strip()


# ------------------------------------------------------------------ MAIN
if __name__ == "__main__":
    system = FinancialAgentSystem()
    task, body, note = system.run(QUERY)
    md_path, html_path = save_report(QUERY, task, body, note)

    print("\n==============================")
    print("FINAL FINANCIAL PIPELINE REPORT")
    print("==============================\n")
    print(body)
    print(f"\nSaved report:\n  {md_path}\n  {html_path}")

    try:
        from IPython.display import Markdown, display
        display(Markdown(body))
    except Exception:
        pass
    sys.stdout.flush()

In [ ]:
try:
    from IPython.display import Markdown, display
    display(Markdown(body))
except Exception:
    pass
sys.stdout.flush()

# Objective

The objective of this report is to develop a rigorous quantitative framework to model the impact of cross-border liquidity shocks on the Value-at-Risk (VaR) of a multi-currency portfolio. The framework should account for the effects of liquidity shocks on the portfolio's returns and provide a comprehensive approach to estimating VaR.

# Assumptions

The following assumptions are made in this framework:

1.  Market returns are normally distributed with a mean of μ and a covariance matrix of Σ.
2.  Liquidity shocks are mean-reverting with jumps and are correlated across borders.
3.  Liquidity cost is linear in trade size and proportional to spread widening and market impact.
4.  The joint distribution of market returns and liquidity shocks is stationary within a regime, and regime switches are captured by stress scenarios.
5.  No arbitrage in liquidity cost: effective price = mid price - cost.

# Methodology

The framework consists of the following steps:

1.  Define the portfolio with positions in multiple currencies and FX rates.
2.  Estimate the market covariance matrix using a factor model.
3.  Model the liquidity shock process using a mean-reverting jump-diffusion process.
4.  Estimate the liquidity-adjusted returns using the liquidity shock process.
5.  Compute the liquidity-adjusted VaR using the liquidity-adjusted returns and the market covariance matrix.

# Key Findings

The key findings of this framework are:

1.  The liquidity-adjusted VaR is a more comprehensive measure of risk that accounts for the effects of liquidity shocks on the portfolio's returns.
2.  The framework provides a rigorous approach to estimating VaR that accounts for the correlations between market returns and liquidity shocks.
3.  The use of a mean-reverting jump-diffusion process to model the liquidity shock process provides a more realistic representation of the dynamics of liquidity shocks.

# Risks and Limitations

The following risks and limitations are associated with this framework:

1.  Model risk: The normal/t assumption underestimates tail dependence during capital controls.
2.  Liquidity proxy risk: Spreads and depth are noisy and not available for all markets.
3.  Correlation breakdown: Σ_{rL} and Σ_L are regime dependent, correlation spikes in crisis.
4.  Non-synchronicity: FX and local markets close at different times, induces artificial correlation.
5.  Market impact non-linearity: MI_c grows super-linearly with size, linear approximation fails for large shocks.

# Next Steps

The next steps are to:

1.  Implement the framework in a programming language such as Python or R.
2.  Test the framework using historical data and compare the results with other VaR models.
3.  Refine the framework by incorporating additional features such as funding liquidity and regulatory risk.

In [ ]:
!ls -ltha /content/drive/MyDrive/reports/*085703*

-rw------- 1 root root 4.1K Nov 15 08:57 /content/drive/MyDrive/reports/financial_report_20261115_085703.html
-rw------- 1 root root 3.2K Nov 15 08:57 /content/drive/MyDrive/reports/financial_report_20261115_085703.md


In [ ]:
from IPython.display import HTML, display
display(HTML(open("/content/drive/MyDrive/reports/financial_report_20261115_085703.html").read()))

In [ ]:
!nvidia-smi

Sun Nov 15 09:04:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   35C    P0             88W /  600W |   87697MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----